# drfn: the Ge'ez ByT5 root analyser in the 2019 root + pattern setup

The 2019 advisor update trained the BPE transformer (about 20M parameters, AGE Ge'ez→Amharic, seeds 11/22/33) with
root and pattern heads on geezRPA labels. This notebook changes only the source of the labels.

* **Step 1.** The Ge'ez ByT5 root analyser (weights from the attached output of `geez-morphology-byt5-training`, seed-1
  books model) reads every distinct Ge'ez word of train + dev once. Its readings become a root + pattern label cache of the
  same size as k256: top 255 roots + `<none>`, top 16 patterns + `<none>`, ranked on train tokens.
* **Step 2.** Runs, in this order:
  1. `labels_both_s11`, `labels_both_s22`, `labels_both_s33`: BPE input, root + pattern heads on analyser labels.
     Compare with *Both labels* (geezRPA) and *Plain BPE* in advisor-update-2019.
  2. `rootin_plain_s11`: BPE pieces plus one analyser root token per word as input, no heads. Compare with *Plain BPE* s11.
     Exploratory.

Inputs to attach: dataset `drfn-analyser-pack`, dataset `geez-morphology-training-inputs`, output of notebook
`geez-morphology-byt5-training`. GPU on, Internet on.

In [ ]:
SEEDS = "11 22 33"     # label runs, in this order
INPUT_RUN = True        # run 4 (root token as input) after the label runs
CACHE_ONLY = False      # True: stop after step 1

In [ ]:
!pip install -q transformers==5.19.0 accelerate==1.15.0 tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat + ' (attach the input that holds it)')
    return hits[0]
pack_src = os.path.dirname(find('run_analyser_runs.sh'))
bundle   = find('geez-morphology_3dc784e.bundle')
weights  = os.path.dirname(find('best_weights_by_val_score/model.safetensors'))
print('pack   ', pack_src); print('bundle ', bundle); print('weights', weights)
fin = os.path.join(os.path.dirname(weights), 'RUN_FINISHED.txt')
print(open(fin).read() if os.path.exists(fin) else '(no RUN_FINISHED.txt)')
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
if not os.path.exists('/tmp/geez_morph'):
    subprocess.run(['git', 'clone', '-q', bundle, '/tmp/geez_morph'], check=True)
    subprocess.run(['git', '-C', '/tmp/geez_morph', 'checkout', '-q', '3dc784eb7c41133712560c33ebdbd7f80d2f0634'], check=True)
print(subprocess.run(['git', '-C', '/tmp/geez_morph', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
print(subprocess.run(['sha256sum', 'train.py'], capture_output=True, text=True).stdout)

In [ ]:
# Step 1: the analyser reads every distinct Ge'ez word once; readings + label cache go to runs/shared/
def stream(cmd, env=None, skip_steps=False):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if skip_steps and (line.startswith('{"step"') or line.startswith("{'step'")): continue
        print(line, end='')
    return p.wait()
if not os.path.exists('runs/shared/analyser_labels.json'):
    if stream([sys.executable, '-u', 'build_analyser_cache.py', '--data-dir', 'data', '--morphology-repo', '/tmp/geez_morph',
               '--model-dir', weights, '--out-dir', 'runs/shared']):
        raise RuntimeError('cache build failed')
else:
    print('label cache exists, kept')

In [ ]:
# What the analyser gave
c = json.load(open('runs/shared/analyser_cache.json', encoding='utf-8')); L = json.load(open('runs/shared/analyser_labels.json', encoding='utf-8'))
print(c['meta']); print(L['metadata'])
print('patterns:', L['pattern_classes'][1:])
print('top roots:', L['root_classes'][1:21])
r = c['readings']
for w in list(r)[::max(1, len(r)//15)][:15]: print(w, '->', r[w])
if CACHE_ONLY: raise RuntimeError('CACHE_ONLY: stopped after step 1')

In [ ]:
# Step 2: the runs (each writes train_log.jsonl, metadata.json, best.pt, last.pt in runs/<name>/)
env = dict(os.environ, SEEDS=SEEDS, INPUT_RUN='1' if INPUT_RUN else '0')
print('exit', stream(['bash', 'run_analyser_runs.sh'], env=env, skip_steps=True))

In [ ]:
# Summary next to the advisor-update-2019 numbers (best dev loss; greedy chrF++ on the first 256 dev sentences at that epoch)
base = {'Plain BPE': {11: (4.4410, 9.32), 22: (4.4267, 10.24), 33: (4.4257, 9.92)},
        'Both labels (geezRPA)': {11: (4.3537, 11.44), 22: (4.3459, 11.41), 33: (4.3566, 11.35)}}
rows = []
for d in sorted(glob.glob('runs/labels_*') + glob.glob('runs/rootin_*')):
    ev = [json.loads(l) for l in open(os.path.join(d, 'train_log.jsonl')) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    rows.append((os.path.basename(d), len(ev), round(b['dev_loss'], 4), round(b['dev_chrf'], 2), b['epoch']))
print('%-22s %6s %10s %7s %5s' % ('run', 'epochs', 'dev_loss', 'chrF++', 'best'))
for r_ in rows: print('%-22s %6d %10.4f %7.2f %5d' % r_)
print(); [print(k, v) for k, v in base.items()]
open('/kaggle/working/summary.txt', 'w').write('\n'.join('\t'.join(map(str, r_)) for r_ in rows) + '\n')